# EDA for `creditcard_2023.csv.zip`

This notebook reads the zipped archive directly, checks the internal CSV member, loads the data with compact dtypes, and runs a focused exploratory data analysis on data quality, class imbalance, transaction amount behavior, and the most useful fraud-related features.

It works from either the current working directory or `MyDrive/thesis` in Colab.


In [ ]:
from pathlib import Path
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


: 

In [ ]:
CANDIDATE_PATHS = [
    Path("creditcard_2023.csv.zip"),
    Path("/content/drive/MyDrive/thesis/creditcard_2023.csv.zip"),
]

DATA_PATH = next((path for path in CANDIDATE_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find creditcard_2023.csv.zip in the current directory or /content/drive/MyDrive/thesis"
    )

with zipfile.ZipFile(DATA_PATH) as archive:
    ARCHIVE_MEMBERS = archive.namelist()

if not ARCHIVE_MEMBERS:
    raise ValueError(f"Archive is empty: {DATA_PATH}")

CSV_MEMBER = ARCHIVE_MEMBERS[0]
print(f"Using archive: {DATA_PATH}")
print(f"Archive members: {ARCHIVE_MEMBERS}")
print(f"CSV member selected: {CSV_MEMBER}")


In [ ]:
dtype_map = {
    "id": "int64",
    "Amount": "float32",
    "Class": "int8",
    **{f"V{i}": "float32" for i in range(1, 29)},
}

with zipfile.ZipFile(DATA_PATH) as archive:
    with archive.open(CSV_MEMBER) as handle:
        df = pd.read_csv(handle, dtype=dtype_map)

memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {df.shape[1]}")
print(f"Approximate in-memory size: {memory_mb:,.2f} MB")

display(df.head())


In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(4),
        "nunique": df.nunique(dropna=False),
    }
).sort_values(["missing", "nunique"], ascending=[False, False])

display(overview)

summary_cols = ["mean", "std", "min", "25%", "50%", "75%", "max"]
display(df.describe(percentiles=[0.25, 0.5, 0.75]).T[summary_cols])


In [ ]:
feature_hash = pd.util.hash_pandas_object(df.loc[:, df.columns != "id"], index=False)
duplicate_feature_rows = int(feature_hash.duplicated().sum())
class_counts = df["Class"].value_counts().sort_index()
class_pct = (class_counts / len(df) * 100).round(4)
fraud_rate = float(class_pct.get(1, 0.0))

quality_summary = pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_id_rows": int(df["id"].duplicated().sum()),
        "duplicate_rows_excluding_id": duplicate_feature_rows,
        "non_fraud_rows": int(class_counts.get(0, 0)),
        "fraud_rows": int(class_counts.get(1, 0)),
        "fraud_rate_pct": fraud_rate,
    }
)

display(quality_summary.to_frame("value"))

plot_labels = ["Non-fraud", "Fraud"]
plot_counts = [int(class_counts.get(0, 0)), int(class_counts.get(1, 0))]
plot_pct = [float(class_pct.get(0, 0.0)), float(class_pct.get(1, 0.0))]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(plot_labels, plot_counts, color=["#4C78A8", "#E45756"])
axes[0].set_yscale("log")
axes[0].set_title("Class Counts (log scale)")
axes[0].set_ylabel("Rows")

axes[1].bar(plot_labels, plot_pct, color=["#4C78A8", "#E45756"])
axes[1].set_title("Class Share (%)")
axes[1].set_ylabel("Percent")

plt.tight_layout()
plt.show()


In [ ]:
amount_summary = df.groupby("Class")["Amount"].agg(["count", "mean", "median", "std", "min", "max"])
amount_summary.index = amount_summary.index.map({0: "Non-fraud", 1: "Fraud"})
display(amount_summary)

plot_df = df.sample(n=min(100_000, len(df)), random_state=42).copy()
plot_df["ClassLabel"] = plot_df["Class"].map({0: "Non-fraud", 1: "Fraud"})
plot_df["LogAmount"] = np.log1p(plot_df["Amount"])

box_df = plot_df.sample(n=min(20_000, len(plot_df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(
    data=plot_df,
    x="LogAmount",
    hue="ClassLabel",
    bins=80,
    stat="density",
    common_norm=False,
    element="step",
    ax=axes[0],
)
axes[0].set_title("log1p(Amount) Distribution by Class")
axes[0].set_xlabel("log1p(Amount)")

sns.boxplot(data=box_df, x="ClassLabel", y="Amount", showfliers=False, ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Amount by Class (log scale, no fliers)")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()


In [ ]:
corr_with_class = (
    df.drop(columns="id")
    .corr(numeric_only=True)["Class"]
    .drop("Class")
    .sort_values(key=lambda series: series.abs(), ascending=False)
)

display(corr_with_class.head(15).to_frame("corr_with_class"))

heatmap_features = []
for feature in corr_with_class.head(12).index.tolist() + ["Amount", "Class"]:
    if feature not in heatmap_features:
        heatmap_features.append(feature)

plt.figure(figsize=(11, 8))
sns.heatmap(
    df[heatmap_features].corr(numeric_only=True),
    cmap="coolwarm",
    center=0,
    annot=False,
    linewidths=0.5,
)
plt.title("Correlation Heatmap for Top Fraud-Linked Features")
plt.tight_layout()
plt.show()


In [ ]:
top_features = corr_with_class.head(6).index.tolist()
fraud_df = df[df["Class"] == 1]
non_fraud_df = df[df["Class"] == 0]
normal_sample_n = min(len(non_fraud_df), max(len(fraud_df) * 5, 20_000))
balanced_plot_df = pd.concat(
    [
        non_fraud_df.sample(n=normal_sample_n, random_state=42),
        fraud_df,
    ],
    axis=0,
).copy()
balanced_plot_df["ClassLabel"] = balanced_plot_df["Class"].map({0: "Non-fraud", 1: "Fraud"})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feature in zip(axes.flat, top_features):
    sns.boxplot(
        data=balanced_plot_df,
        x="ClassLabel",
        y=feature,
        showfliers=False,
        ax=ax,
    )
    ax.set_title(f"{feature} by Class")
    ax.set_xlabel("")

for ax in axes.flat[len(top_features):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
positive_corr = corr_with_class.sort_values(ascending=False).head(5)
negative_corr = corr_with_class.sort_values(ascending=True).head(5)

print("Top positive correlations with Class:")
display(positive_corr.to_frame("correlation"))

print("Top negative correlations with Class:")
display(negative_corr.to_frame("correlation"))

print("Recommended next steps:")
print("1. Use stratified splits because the dataset is highly imbalanced.")
print("2. Track ROC-AUC, PR-AUC, recall, and precision instead of raw accuracy alone.")
print("3. Compare models on the strongest features first, then test richer feature combinations.")
print("4. Apply class weighting, focal loss, or resampling if you build supervised fraud models.")
